# Signal Features and Fault Clustering Tutorial #
This notebook is the second stage of a predictive maintenance system. Stage 1 flagged unusual readings. Here we turn those flagged readings into candidate groups, to see whether they share a common fault.<br>
<br>
We use the real SKAB benchmark, a water pump testbed with labeled faults. The labels are used only at the end, to check the groups. The groups themselves come from the data alone.<br>
<br>
The notebook covers:
  - Signal features: time-domain statistics and frequency-band energy
  - Sliding windows over a stream
  - Reducing features with PCA and visualizing them
  - Clustering with K-means, Gaussian mixtures, DBSCAN, and HDBSCAN
  - Choosing the number of clusters with silhouette scores and BIC
  - Stability: do the clusters survive resampling?
  - Checking clusters against known faults
  - Novelty detection: flagging windows that fit no known group
<br>

## Theory ##

Stage 1 answers "is this reading unusual?" Stage 2 answers "what kind of unusual is it?" Several unusual readings that look alike probably have the same cause. Grouping them is the first step toward a fault library.

**Signal features**

Raw sensor readings are long and noisy. Features summarize a window of readings into a short list of numbers that describe its shape:

  - **Standard deviation:** how much the signal moves
  - **Kurtosis:** how heavy the tails are. A signal that is quiet most of the time with sharp bursts has high kurtosis. A normal bell-shaped signal has kurtosis near 0 (this is "excess" kurtosis).
  - **Crest factor:** the largest peak divided by the typical size. Impacts, such as a bearing knock, make it jump.
  - **Frequency-band energy:** the share of the signal's energy in each frequency band. A fault that adds a vibration at a specific frequency moves energy from one band to another.

$$
\text{crest} = \frac{\max_t |x_t - \bar{x}|}{\sigma_x}
$$

**Clustering**

Clustering groups points so that points in a group are similar and points in different groups are not. It needs no labels. Common methods:

  - **K-means:** chooses $k$ centres and assigns each point to the nearest one. Groups are round and of similar size.
  - **Gaussian mixture model (GMM):** assumes each group is a bell-shaped cloud with its own size and orientation. It gives a probability for each point, not just a label.
  - **DBSCAN:** finds dense regions and marks sparse points as noise. It needs a neighbourhood size $\varepsilon$.
  - **HDBSCAN:** a version of DBSCAN that works across many densities and needs fewer settings.

**Checking the clusters**

Clustering always returns groups, even when there are none. Three checks help:
  - **Silhouette score:** for each point, how much closer it is to its own group than to the next nearest group. Ranges from -1 to 1. Higher is better.
  - **BIC:** for a GMM, a score that rewards fit and penalizes extra parameters. Lower is better.
  - **Stability:** if you cluster a random part of the data, do you get the same groups? Groups that appear only once are probably noise.

**Novelty detection**

Fit the groups on known faults. A new window that fits none of them well is a candidate for a new failure mode. This is the link to Stage 3, where a person decides what the new group is.

**Real-world use**

  - **Wind turbines:** vibration signatures of gearbox wear, bearing damage, and blade imbalance form separate clusters. A maintenance team can name a cluster after an inspection finds its cause.
  - **Pumps and compressors:** cavitation, which is bubbles forming and collapsing in the fluid, can produce sharp impacts. Features that capture impacts and frequency bands may help separate it, though in the results below it still shares a group with rotor imbalance.
  - **Fleets of similar machines:** grouping faults across many units shows which failure modes are common, and which units behave differently.

## The SKAB Dataset ##

The Skoltech Anomaly Benchmark (SKAB) records a small water circulation pump with eight sensors, sampled once per second. Each file is one experiment:

  - **anomaly-free:** normal operation
  - **valve1 and valve2:** the inlet and outlet valves are closed, which changes the flow and pressure
  - **other:** fluid leaks and additions, rotor imbalance, water-level changes, cavitation, and hot water

The benchmark is published under the GPL-3.0 licence, from the [SKAB repository](https://github.com/waico/SKAB). The files are downloaded below, the first time you run the notebook.

### Download the data ###

In [ ]:
%matplotlib inline
import csv
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.cluster import KMeans, DBSCAN, HDBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors

# Fixed seed, so every run gives the same results
rng = np.random.default_rng(42)

In [ ]:
# Download the SKAB files from GitHub once, into a local data folder
BASE_URL = "https://raw.githubusercontent.com/waico/SKAB/master/data/"
DATA_DIR = os.path.join("data", "skab")

# Normal runs, two valve-closing families, and the 14 'other' experiments
FILES = (["anomaly-free/anomaly-free.csv"]
         + [f"valve1/{i}.csv" for i in range(16)]
         + [f"valve2/{i}.csv" for i in range(4)]
         + [f"other/{i}.csv" for i in range(1, 15)])

for rel in FILES:
    local = os.path.join(DATA_DIR, *rel.split("/"))
    os.makedirs(os.path.dirname(local), exist_ok=True)
    if not os.path.exists(local):
        urllib.request.urlretrieve(BASE_URL + rel, local)

print(f"{len(FILES)} files ready in {DATA_DIR}")

### Load and look at one experiment ###

Each file is semicolon-separated, with a timestamp, eight sensor columns, and two label columns: `anomaly` and `changepoint`. The anomaly-free file has no label columns.

In [ ]:
def load_skab(path):
    # Read one SKAB file: semicolon-separated, with a header row and a timestamp column
    with open(path, encoding="utf-8") as f:
        reader = csv.reader(f, delimiter=";")
        header = next(reader)
        rows = list(reader)
    names = header[1:]
    data = np.array([[float(v) for v in row[1:]] for row in rows])
    sensor_names = names[:8]           # the eight sensor columns come first
    sensors = data[:, :8]
    if "anomaly" in names:
        anomaly = data[:, names.index("anomaly")]
    else:
        anomaly = np.zeros(len(data))  # the anomaly-free file has no label column
    return sensor_names, sensors, anomaly

SENSOR_NAMES, sensors, anomaly = load_skab(os.path.join(DATA_DIR, "valve1", "3.csv"))
print("samples:", len(sensors), " anomalous samples:", int(anomaly.sum()))
print("sensors:", SENSOR_NAMES)

In [ ]:
# Plot four sensors from the same experiment, with the labeled anomaly shaded
show = ["Accelerometer1RMS", "Current", "Pressure", "Temperature"]
fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
for ax, name in zip(axes, show):
    j = SENSOR_NAMES.index(name)
    ax.plot(sensors[:, j], lw=0.8)
    ax.fill_between(np.arange(len(anomaly)), sensors[:, j].min(), sensors[:, j].max(),
                    where=anomaly > 0, color="red", alpha=0.15)
    ax.set_ylabel(name, fontsize=8)
axes[0].set_title("valve1/3: the red band is the labeled anomaly")
axes[-1].set_xlabel("time (seconds)")
plt.tight_layout()
plt.show()

## Signal Features ##

We slide a one-minute window across each experiment, moving 30 seconds at a time. For each window and each sensor we compute six features: the standard deviation, kurtosis, crest factor, and the share of energy in three frequency bands.

A window is labeled anomalous when at least half of its samples are anomalous. Stage 1 would flag these windows, so they are the input to clustering.

**Real-world example:** an engineer listening to a pump does not hear every sample. They hear the rhythm, the sharpness of knocks, and whether a hum has changed pitch. Features are a numeric version of that listening.

In [ ]:
WINDOW = 60   # one minute, at one sample per second
STEP = 30     # slide the window 30 seconds at a time

def window_features(w):
    # Features for one window of one sensor's signal
    centred = w - w.mean()
    std = centred.std()
    kurt = stats.kurtosis(w)                         # excess kurtosis: heaviness of the tails
    crest = np.max(np.abs(centred)) / (std + 1e-12)  # largest peak relative to typical size
    # Share of the signal's energy in three frequency bands
    power = np.abs(np.fft.rfft(centred)) ** 2
    freqs = np.fft.rfftfreq(len(w), d=1.0)           # one sample per second
    total = power.sum() + 1e-12
    bands = [(0.0, 0.05), (0.05, 0.15), (0.15, 0.5)]
    shares = [power[(freqs >= lo) & (freqs < hi)].sum() / total for lo, hi in bands]
    return [std, kurt, crest, *shares]

FEATURES_PER_SENSOR = ["std", "kurtosis", "crest", "band 0-0.05 Hz", "band 0.05-0.15 Hz", "band 0.15-0.5 Hz"]
FEATURE_NAMES = [f"{s}: {f}" for s in SENSOR_NAMES for f in FEATURES_PER_SENSOR]

def make_windows(sensors, anomaly):
    # Slide a window over one experiment, and label it anomalous if at least half its samples are
    feats, labels = [], []
    for start in range(0, len(sensors) - WINDOW + 1, STEP):
        stop = start + WINDOW
        window = sensors[start:stop]
        feats.append(np.concatenate([window_features(window[:, j]) for j in range(window.shape[1])]))
        labels.append(anomaly[start:stop].mean() >= 0.5)
    return np.array(feats), np.array(labels)

In [ ]:
def family_of(rel):
    # Fault family for one file. Used only to check the clusters at the end, never to build them
    if rel.startswith("anomaly-free"):
        return "normal"
    if rel.startswith("valve1"):
        return "valve 1 closed"
    if rel.startswith("valve2"):
        return "valve 2 closed"
    idx = int(rel.split("/")[1].split(".")[0])
    if idx <= 4:
        return "leak or addition"
    if idx <= 9:
        return "rotor imbalance"
    if idx <= 11:
        return "water level"
    if idx <= 13:
        return "cavitation"
    return "hot water"

# Build the feature table for every experiment
X_all, y_all, fam_all, exp_all = [], [], [], []
for rel in FILES:
    _, sensors_i, anomaly_i = load_skab(os.path.join(DATA_DIR, *rel.split("/")))
    feats, labels = make_windows(sensors_i, anomaly_i)
    X_all.append(feats)
    y_all.append(labels)
    fam_all += [family_of(rel)] * len(feats)
    exp_all += [rel] * len(feats)

X_all = np.vstack(X_all)
y_all = np.concatenate(y_all)
fam_all = np.array(fam_all)
exp_all = np.array(exp_all)

print("windows:", len(X_all), " features per window:", X_all.shape[1])
print("anomalous windows:", int(y_all.sum()))

In [ ]:
# Plot kurtosis and crest factor of the current sensor for one experiment
s_idx = FEATURE_NAMES.index("Current: kurtosis")
c_idx = FEATURE_NAMES.index("Current: crest")
rel = "valve1/3.csv"
mask = exp_all == rel
flags = y_all[mask]
x = np.arange(mask.sum())

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
for ax, idx, title in [(axes[0], s_idx, "kurtosis"), (axes[1], c_idx, "crest factor")]:
    ax.plot(x, X_all[mask, idx], lw=1)
    ax.fill_between(x, X_all[mask, idx].min(), X_all[mask, idx].max(), where=flags,
                    color="red", alpha=0.15)
    ax.set_title(f"Current, {title}, per window ({rel})")
axes[-1].set_xlabel("window number")
plt.tight_layout()
plt.show()

## Reducing and Visualizing the Features ##

We have 48 features per window, and many of them move together. Principal component analysis (PCA) compresses them into a few directions that keep most of the variation. We fit PCA on the anomalous windows only, because those are the data we cluster.

Standard deviation and crest factor are positive and skewed, with values that spread over orders of magnitude. We take their logs first. Then every feature is standardized using the normal anomaly-free windows as the reference, so each one is measured in the same units of normal variation. Without the log, a few windows with extreme values stretched the scale so far that the clustering results were distorted.

In [ ]:
# Standard deviation and crest factor are positive and heavily skewed, so take their logs.
# No window has a zero value for either, so the log is safe
LOG_FEATURES = [i for i, n in enumerate(FEATURE_NAMES) if n.endswith(": std") or n.endswith(": crest")]
X_model = X_all.copy()
X_model[:, LOG_FEATURES] = np.log(X_model[:, LOG_FEATURES])

# Standardize with the normal reference windows (the anomaly-free experiment)
normal_mask = fam_all == "normal"
ref_mean = X_model[normal_mask].mean(axis=0)
ref_sd = X_model[normal_mask].std(axis=0) + 1e-12
Z_all = (X_model - ref_mean) / ref_sd

# Keep only the anomalous windows, as a Stage 1 detector would flag them
Z = Z_all[y_all]
fam = fam_all[y_all]
exp_id = exp_all[y_all]
print("anomalous windows to cluster:", len(Z))
print(dict(zip(*np.unique(fam, return_counts=True))))

In [ ]:
def fit_pca(Z, var_kept=0.90):
    # Keep the fewest principal components that explain var_kept of the variance
    centre = Z.mean(axis=0)
    _, S, Vt = np.linalg.svd(Z - centre, full_matrices=False)
    explained = np.cumsum(S ** 2) / np.sum(S ** 2)
    k = int(np.searchsorted(explained, var_kept) + 1)
    return centre, Vt[:k], explained

centre, V, explained = fit_pca(Z)
P = (Z - centre) @ V.T
print("components kept:", V.shape[0], " variance explained:", round(explained[V.shape[0] - 1], 3))

In [ ]:
# Plot the first two principal components, coloured by the known fault family
_, _, Vt2 = np.linalg.svd(Z - centre, full_matrices=False)
P2 = (Z - centre) @ Vt2[:2].T

fig, ax = plt.subplots(figsize=(8, 6))
for name in np.unique(fam):
    m = fam == name
    ax.scatter(P2[m, 0], P2[m, 1], s=14, label=name)
ax.set_xlabel("principal component 1")
ax.set_ylabel("principal component 2")
ax.set_title("Anomalous windows, coloured by known fault family")
ax.legend(fontsize=8)
plt.show()

The clustering never sees the family colours. Most windows sit in one dense area, with valve 1 making up much of it. A few windows lie far from that area, and they stretch the axes. Families overlap heavily in these two dimensions, so a two-dimensional picture alone cannot tell us how many groups there are.

## Choosing the Number of Clusters ##

K-means needs $k$ in advance. Two checks guide the choice. The silhouette score measures how well each window fits its own group. A Gaussian mixture gives a BIC score that balances fit against the number of parameters.

In [ ]:
ks = range(2, 11)
silhouettes, bics = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(P)
    silhouettes.append(silhouette_score(P, km.labels_))
    gmm = GaussianMixture(n_components=k, covariance_type="full", random_state=42, n_init=3).fit(P)
    bics.append(gmm.bic(P))

best_k_sil = list(ks)[int(np.argmax(silhouettes))]
best_k_bic = list(ks)[int(np.argmin(bics))]
for k, s, b in zip(ks, silhouettes, bics):
    print(f"k={k:2d}  silhouette={s:.3f}  BIC={b:9.1f}")
print("best k by silhouette:", best_k_sil, " best k by BIC:", best_k_bic)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(ks), silhouettes, "o-")
axes[0].set_title("K-means silhouette (higher is better)")
axes[0].set_xlabel("k")
axes[1].plot(list(ks), bics, "o-", color="tab:orange")
axes[1].set_title("GMM BIC (lower is better)")
axes[1].set_xlabel("k")
plt.tight_layout()
plt.show()

The two checks disagree. The silhouette score is highest at $k=2$, at 0.34, and falls to about 0.2 by $k=6$. BIC keeps finding better fits up to $k=7$. A silhouette peak at two groups is a common sign that the data have one main split, with finer structure underneath. We carry both values forward and check them against stability and the known families.

## Density-Based Clustering ##

K-means assumes round groups of similar size. DBSCAN and HDBSCAN find groups of any shape and mark sparse points as noise. A useful noise label matters in maintenance, because a window that fits no group needs a person to look at it.

**Real-world example:** a fleet of pumps produces a few windows that are unlike anything else. DBSCAN labels them as noise, so they are not forced into a fault group they do not belong to.

In [ ]:
# Choose DBSCAN's neighbourhood size from the distance to each point's 5th nearest neighbour
nn = NearestNeighbors(n_neighbors=5).fit(P)
kdist = np.sort(nn.kneighbors(P)[0][:, -1])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(kdist)
ax.set_title("5th-nearest-neighbour distance, sorted (look for the bend)")
ax.set_xlabel("windows, sorted")
ax.set_ylabel("distance")
plt.show()

eps = float(np.quantile(kdist, 0.85))   # a value near the bend of the curve
db_labels = DBSCAN(eps=eps, min_samples=5).fit_predict(P)
hdb_labels = HDBSCAN(min_cluster_size=10, min_samples=3).fit_predict(P)

for name, labels in [("DBSCAN", db_labels), ("HDBSCAN", hdb_labels)]:
    groups = sorted(set(labels) - {-1})
    print(f"{name}: {len(groups)} groups, {np.sum(labels == -1)} noise windows")

DBSCAN finds four groups and labels 35 windows as noise, with the neighbourhood size set at the 85th percentile of 5th-nearest-neighbour distances. HDBSCAN finds two groups and labels 11 windows as noise. The two methods disagree, and that is worth noticing: a density-based result depends on settings, and the groups can change when the settings change.

## Stability ##

A cluster that appears only once may be an accident of the data. To test stability, cluster random 70% subsets of the windows and compare the groupings on the windows they share. The adjusted Rand index (ARI) measures agreement: 1 means identical groupings, and 0 means no better than chance.

In [ ]:
def stability(fit_model, P, n_runs=20):
    # fit_model(X) returns a fitted model with a predict method
    # Agreement between two fits on random 70% subsets, measured on the windows both subsets share
    scores = []
    for _ in range(n_runs):
        idx_a = rng.choice(len(P), size=int(0.7 * len(P)), replace=False)
        idx_b = rng.choice(len(P), size=int(0.7 * len(P)), replace=False)
        shared = np.intersect1d(idx_a, idx_b)
        labels_a = fit_model(P[idx_a]).predict(P[shared])
        labels_b = fit_model(P[idx_b]).predict(P[shared])
        scores.append(adjusted_rand_score(labels_a, labels_b))
    return np.mean(scores), np.std(scores)

kmeans_fit = lambda X: KMeans(best_k_sil, n_init=10, random_state=42).fit(X)
gmm_fit = lambda X: GaussianMixture(best_k_bic, covariance_type="full", random_state=42, n_init=3).fit(X)
print("K-means, k =", best_k_sil, ": mean ARI, std =", np.round(stability(kmeans_fit, P), 3))
print("GMM, k =", best_k_bic, ": mean ARI, std =", np.round(stability(gmm_fit, P), 3))

K-means at $k=2$ is very stable: two fits on different random subsets agree almost perfectly (ARI 0.99). The GMM at $k=7$ is much less stable (ARI 0.54), which matches the lower BIC preference. Stable groups are not necessarily correct groups, so we check them next.

## Checking Against Known Faults ##

Now we use the labels, which the clustering never saw. For each method we ask two questions:
  - **Do the groups match the known fault families?** The adjusted Rand index compares them.
  - **Are the groups just separate experiments?** Each experiment file is its own run, and runs can differ for reasons unrelated to faults. If the groups match files better than families, the clustering has found runs, not faults.

In [ ]:
gmm_labels = GaussianMixture(best_k_bic, covariance_type="full", random_state=42, n_init=3).fit(P).predict(P)
km_labels = KMeans(best_k_sil, n_init=10, random_state=42).fit_predict(P)

def crosstab(a, b):
    # Counts of each (a, b) pair, printed as a table
    rows, cols = np.unique(a), np.unique(b)
    print(" " * 20 + "".join(f"{str(c):>8}" for c in cols))
    for r in rows:
        counts = [int(np.sum((a == r) & (b == c))) for c in cols]
        print(f"{str(r):<20}" + "".join(f"{v:>8}" for v in counts))

def purity(labels, truth):
    # Share of windows that belong to the most common family in their group
    total = 0
    for g in np.unique(labels):
        _, counts = np.unique(truth[labels == g], return_counts=True)
        total += counts.max()
    return total / len(labels)

for name, labels in [("K-means", km_labels), ("GMM", gmm_labels), ("HDBSCAN", hdb_labels)]:
    print(f"{name}: ARI vs family = {adjusted_rand_score(fam, labels):.3f}, "
          f"ARI vs experiment file = {adjusted_rand_score(exp_id, labels):.3f}, "
          f"purity vs family = {purity(labels, fam):.3f}")

print("\nK-means groups vs fault families:")
crosstab(fam, km_labels)

K-means at $k=2$ matches the fault families moderately well (ARI 0.45, purity 0.63). It barely matches the experiment files (ARI 0.04), so the groups reflect fault behaviour more than individual runs. That is the result we wanted.

The table shows that the split is coarse. One group holds nearly all the valve 1 and valve 2 windows, plus most leak windows. The other holds cavitation, hot water, rotor imbalance, and water level. Cavitation and rotor imbalance end up together, so these features do not separate every fault. The GMM and HDBSCAN groups match the families less well (ARI 0.30 and 0.06).

In [ ]:
# Plot the windows in two dimensions with t-SNE, coloured by fault family
emb = TSNE(n_components=2, perplexity=30, init="pca", random_state=42).fit_transform(P)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name in np.unique(fam):
    m = fam == name
    axes[0].scatter(emb[m, 0], emb[m, 1], s=12, label=name)
axes[0].set_title("coloured by known fault family")
axes[0].legend(fontsize=7)

for g in np.unique(gmm_labels):
    m = gmm_labels == g
    axes[1].scatter(emb[m, 0], emb[m, 1], s=12, label=f"group {g}")
axes[1].set_title("coloured by GMM group")
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

In the t-SNE picture the families form islands. Valve 1 is the large island on the left, leak windows form a separate island in the middle, and rotor imbalance, hot water, and water level sit together at upper right. Cavitation is a small island at lower right. The GMM groups mostly follow these islands. The two-dimensional picture is a guide to the structure, not a proof of it, because t-SNE distorts distances.

## Novelty Detection ##

The groups above are fitted on known faults. In a live system, a new fault can appear that matches none of them. To simulate this, we hold out one whole fault family, fit a GMM on the others, and ask whether the held-out windows look unfamiliar.

The threshold comes from the training windows. We use the 5th percentile of their log-likelihood, so about 5% of known windows would be flagged. A held-out family should be flagged far more often than that. We repeat this for every family in turn.

**Real-world example:** a new bearing defect develops in one pump in a fleet. Its vibration matches no known fault group, so it is flagged for an engineer, who decides whether it is a new failure mode.

In [ ]:
# Hold out each fault family in turn: fit on the others, then score the held-out windows
N_NOVEL = best_k_bic
print(f"{'held-out family':<20}{'windows':>9}{'flagged as novel':>19}")
for held in np.unique(fam):
    train = fam != held
    gmm_novel = GaussianMixture(N_NOVEL, covariance_type="full", random_state=42, n_init=3).fit(P[train])
    threshold = np.quantile(gmm_novel.score_samples(P[train]), 0.05)
    held_scores = gmm_novel.score_samples(P[~train])
    print(f"{held:<20}{len(held_scores):>9}{np.mean(held_scores < threshold):>19.1%}")

The held-out check gives a clear signal, though a noisy one. Every held-out family is flagged more often than the 5% baseline. Rates range from 40% for hot water (10 windows) to 100% for leak or addition (44 windows). Valve 1, the largest family, is flagged 83% of the time. Valve 2 is flagged only 41% of the time, probably because it looks like valve 1, which stayed in training. Small families make every one of these numbers unstable. One threshold cannot reliably separate a new fault from an unusual version of a known one.

## Summary ##

  - **Features turn windows into numbers.** Standard deviation, kurtosis, crest factor, and frequency-band energy describe the shape of a signal, not just its level.
  - **Scale matters.** Skewed features need a transform before clustering. Without the log, a few extreme windows distorted the results.
  - **The checks disagree, and that is information.** Silhouette favours two groups. BIC favours seven. The two-group split is the one that is stable.
  - **Stable is not the same as correct.** K-means at two groups matched the fault families moderately (ARI 0.45) and the experiment files poorly (ARI 0.04). The groups reflect fault behaviour, but they are coarse.
  - **Density methods depend on their settings.** DBSCAN and HDBSCAN found different numbers of groups and different noise counts.
  - **Novelty detection gives Stage 3 its input.** Every held-out family was flagged above the baseline, but the rates varied widely. The windows flagged as novel are the ones a person should look at and label next.
  - **The groups are candidates, not fault names.** Stage 3 needs a person to name them.

## Further Reading ##
  - SKAB repository and paper: https://github.com/waico/SKAB
  - Ester et al. (1996), *A density-based algorithm for discovering clusters*, for DBSCAN
  - Campello, Moulavi and Sander (2013), *Density-based clustering based on hierarchical density estimates*, for HDBSCAN
  - scikit-learn clustering guide: https://scikit-learn.org/stable/modules/clustering.html
  - Hastie, Tibshirani and Friedman, *The Elements of Statistical Learning*, chapter 14, for clustering and the GMM